## Customer Spend Summary — customer_spend_eur

Purpose: calculate the total amount each customer spent, expressed in EUR.

| Step | What it does | Result |
|------|--------------|--------|
| 1 | Join clean orders to the exchange rates on the order's fx_reference_date and currency | rates matched to each order |
| 2 | Convert each line to EUR (line_total ÷ rate; EUR stays as-is, RON divided by ~5.25) | amounts in EUR |
| 3 | Keep only completed orders (refunds are not "spent") | refunded orders excluded |
| 4 | Sum the EUR amounts per customer, ranked highest first | one row per customer |

**Decision:** only `completed` orders count as spend — a refund returns the money, so it isn't
"spent." (Including refunds would give 1,880 customers / €749,116.15; we did not use that.)

**Result:** 1,866 customers · total spend €713,984.79.

### Conclusion
Every customer's spending is now shown in a single currency (EUR), so customers paying in RON
and EUR can be compared fairly. Each order was converted using the exchange rate for its own
reference date, and only real (completed) sales were counted. The result is a clean, ranked list
of how much each customer actually spent.


In [1]:
%%sql
-- Sums each customer's spend, converting RON orders to EUR using the exchange rate for that order's fx_reference_date. 
-- Only completed orders count as spend (refunds are excluded).
-- Total amount each customer spent, converted to EUR.
-- The project asks for spend per customer in EUR; RON orders are converted using the
--       exchange rate for that order's fx_reference_date.
-- Line_total / rate_to_eur  ->  RON amount / (RON per 1 EUR) = EUR amount.
--      For EUR orders the rate is 1.0, so the amount stays the same.
-- One row per customer with their total EUR spend, highest first.
CREATE OR REPLACE TABLE customer_spend_eur AS
SELECT
  o.customer_id,
  ROUND(SUM(o.line_total / f.rate_to_eur), 2) AS total_spend_eur
FROM orders_clean o
JOIN fx_rates f
  ON  f.fx_date  = o.fx_reference_date
  AND f.currency = o.currency
WHERE o.status = 'completed'          -- count only completed orders (refunds are not "spent")
GROUP BY o.customer_id
ORDER BY total_spend_eur DESC

StatementMeta(, f171cf0e-fde1-46ba-80ff-1a3e2571e333, 2, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [2]:
%%sql
-- Number of customers and total EUR across everyone.
SELECT COUNT(*) AS customers, ROUND(SUM(total_spend_eur), 2) AS grand_total_eur
FROM customer_spend_eur

StatementMeta(, f171cf0e-fde1-46ba-80ff-1a3e2571e333, 3, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>